# 04 · Demo

1. A **Gradio web app** with a public link (works on your phone too): upload a dark photo and get the enhanced version, a before/after slider, and the inference time.
2. **Video**: enhance a video file and save a side-by-side result with an FPS overlay.

You need a trained checkpoint (notebook 01).

## Setup
Runs in about 1–2 minutes. It mounts Google Drive, gets the latest code, installs the requirements, and restores the datasets from the zip cache on your Drive. Run `00_setup_and_data.ipynb` once beforehand.

In [ ]:
REPO_URL = "https://github.com/Asad3404/LUMA.git"
BRANCH = "claude/attention-unet-low-light-ay0gpv"   # change to "main" once merged
DRIVE_ROOT = "/content/drive/MyDrive/lowlight-attn-unet"
DATA_ROOT = "/content/data"
REPO_DIR = "/content/LUMA"

from google.colab import drive
drive.mount("/content/drive")
import os
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --format="code version: %h  %s"
!pip install -q -r requirements.txt 
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python data/download.py --datasets lol_v1 --data_root {DATA_ROOT} --cache_dir {DRIVE_ROOT}/datasets --results_dir {DRIVE_ROOT}/results/data

In [ ]:
CKPT = f"{DRIVE_ROOT}/runs/ours_lolv1/best.pth"
import os; assert os.path.exists(CKPT), "train the model first (notebook 01)"

## Web app
Click the `https://….gradio.live` link printed below. The link stays alive while this cell runs (up to 72 h). Stop the cell to shut the app down.

In [ ]:
!python demo/app.py --checkpoint {CKPT} --examples {DATA_ROOT}/lol_v1/test/low --share

## Video
Upload a dark video (or put one on your Drive and set `VIDEO`). The result is saved next to it on Drive.

In [ ]:
VIDEO = ""   # e.g. f"{DRIVE_ROOT}/videos/night.mp4"; leave empty to upload one now
if not VIDEO:
    from google.colab import files
    up = files.upload()
    VIDEO = list(up)[0]
OUT = os.path.splitext(VIDEO)[0] + "_enhanced.mp4"
if not OUT.startswith("/content/drive"):
    OUT = f"{DRIVE_ROOT}/results/" + os.path.basename(OUT)
!python demo/video.py --checkpoint {CKPT} --input "{VIDEO}" --output "{OUT}" --max_side 1280
print("saved:", OUT)

In [ ]:
# Preview the first seconds in the notebook (re-encoded to H.264 so the browser can play it)
!ffmpeg -y -loglevel error -i "{OUT}" -t 8 -vcodec libx264 -pix_fmt yuv420p /content/preview.mp4
from IPython.display import Video
Video("/content/preview.mp4", embed=True, width=900)